In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/v8_clean.csv
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/349_IM-1697-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/607_IM-2196-1001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/2832_IM-1249-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/699_IM-2263-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/1931_IM-0602-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/947_IM-2442-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/2932_IM-1335-1001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/291_IM-1313-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/1790_IM-0515-1001.dcm.png
/kaggle/input/datasets/diplailai/iu-xray-384-cleaned/images_384/3489_IM-1696-2001.dcm.png
/kaggle/input/datasets/diplailai/iu-xra

In [2]:
import os

os.environ["MODE"] = "probe"
os.environ["EMBED_CONFIG"] = "vit_bioclinicalbert"

os.environ["BATCH_SIZE"] = "6"
os.environ["GRAD_ACCUM"] = "5"

os.environ["EPOCHS"] = "3"
os.environ["EVAL_EVERY"] = "1"
os.environ["FREEZE_EP"] = "1"
os.environ["CLUSTER_START"] = "2"
os.environ["CLINICAL_START"] = "2"
os.environ["NUM_WORKERS"] = "2"

os.environ["SEED"] = "42"
os.environ["SPLIT_SEED"] = "42"
os.environ["RESUME_MODE"] = "off"

In [3]:
# Kaggle IU-Xray controlled dual-GPU runner
# Copy-paste this whole file into one Kaggle Notebook cell.
# Add Kaggle inputs:
#   1) IU-Xray data dataset, e.g. /kaggle/input/iu-xray-384-cleaned
#   2) Code zip dataset containing iu_xray_kaggle_code_controlled.zip
# Optional: add previous output zip/checkpoint dataset and set RESUME_MODE="auto".

import glob
import json
import os
import pathlib
import shutil
import subprocess
import time
import zipfile

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# =========================
# EDIT THESE SETTINGS ONLY
# =========================
MODE = os.environ.get("MODE", "probe")  # probe or true_run
EMBED_CONFIG = os.environ.get("EMBED_CONFIG", "swin_bioclinicalbert")
BATCH_SIZE = os.environ.get("BATCH_SIZE", "4")
GRAD_ACCUM = os.environ.get("GRAD_ACCUM", "8")
NUM_WORKERS = os.environ.get("NUM_WORKERS", "2")
EPOCHS = os.environ.get("EPOCHS", "3" if MODE == "probe" else "100")
EVAL_EVERY = os.environ.get("EVAL_EVERY", "1" if MODE == "probe" else "5")
FREEZE_EP = os.environ.get("FREEZE_EP", "1" if MODE == "probe" else "5")
CLUSTER_START = os.environ.get("CLUSTER_START", "2" if MODE == "probe" else "12")
CLINICAL_START = os.environ.get("CLINICAL_START", "2" if MODE == "probe" else "15")
SEED = os.environ.get("SEED", "42")
SPLIT_SEED = os.environ.get("SPLIT_SEED", "42")
RESUME_MODE = os.environ.get("RESUME_MODE", "auto")  # auto, off, or explicit checkpoint path
CODE_ZIP_NAME = os.environ.get("CODE_ZIP_NAME", "iu_xray_kaggle_code_controlled.zip")
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input/iu-xray-384-cleaned")

CONFIGS = {
    "swin_bioclinicalbert": {
        "image_model": "microsoft/swinv2-base-patch4-window12to24-192to384-22kto1k-ft",
        "text_model": "emilyalsentzer/Bio_ClinicalBERT",
    },
    "vit_bioclinicalbert": {
        "image_model": "google/vit-base-patch16-384",
        "text_model": "emilyalsentzer/Bio_ClinicalBERT",
    },
    "convnext_bioclinicalbert": {
        "image_model": "facebook/convnext-tiny-384",
        "text_model": "emilyalsentzer/Bio_ClinicalBERT",
    },
    "swin_pubmedbert": {
        "image_model": "microsoft/swinv2-base-patch4-window12to24-192to384-22kto1k-ft",
        "text_model": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    },
}

if EMBED_CONFIG not in CONFIGS:
    raise ValueError(f"Unknown EMBED_CONFIG={EMBED_CONFIG}. Expected one of {sorted(CONFIGS)}")

RUN_ID = f"{EMBED_CONFIG}_{MODE}_seed{SEED}_split{SPLIT_SEED}"
REPO_DIR = pathlib.Path("/kaggle/working/finetune_IU_Xray")
RUN_ROOT = pathlib.Path("/kaggle/working/runs") / RUN_ID
LOG_ROOT = pathlib.Path("/kaggle/working/logs")
OUTPUT_ZIP = pathlib.Path("/kaggle/working") / f"{RUN_ID}_outputs.zip"

print("=== Run config ===")
for key, value in {
    "MODE": MODE,
    "EMBED_CONFIG": EMBED_CONFIG,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM": GRAD_ACCUM,
    "EPOCHS": EPOCHS,
    "EVAL_EVERY": EVAL_EVERY,
    "FREEZE_EP": FREEZE_EP,
    "CLUSTER_START": CLUSTER_START,
    "CLINICAL_START": CLINICAL_START,
    "SEED": SEED,
    "SPLIT_SEED": SPLIT_SEED,
    "RESUME_MODE": RESUME_MODE,
    "RUN_ID": RUN_ID,
}.items():
    print(f"{key}: {value}")
print("CONFIG:", json.dumps(CONFIGS[EMBED_CONFIG], indent=2))

print("\n=== Kaggle inputs ===")
for p in sorted(glob.glob("/kaggle/input/*")):
    print("\n", p)
    try:
        print(os.listdir(p)[:40])
    except Exception as exc:
        print("  cannot list:", exc)

print("\n=== Setup code ===")
zip_candidates = sorted(glob.glob(f"/kaggle/input/**/{CODE_ZIP_NAME}", recursive=True))
legacy_zip_candidates = sorted(glob.glob("/kaggle/input/**/iu_xray_kaggle_code_smoke_resume.zip", recursive=True))
all_zip_candidates = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
repo_input_candidates = [p for p in sorted(glob.glob("/kaggle/input/**", recursive=True)) if os.path.isdir(p) and os.path.exists(os.path.join(p, "train_ablation.py"))]

print("expected code zip name:", CODE_ZIP_NAME)
print("matching code zips:", zip_candidates)
print("legacy code zips:", legacy_zip_candidates)
print("all input zips:", all_zip_candidates[:50])
print("repo directory candidates:", repo_input_candidates[:20])

shutil.rmtree(REPO_DIR, ignore_errors=True)
REPO_DIR.mkdir(parents=True, exist_ok=True)

def zip_contains_repo(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = set(zf.namelist())
            return "train_ablation.py" in names or any(name.endswith("/train_ablation.py") for name in names)
    except zipfile.BadZipFile:
        return False

scanned_repo_zips = [p for p in all_zip_candidates if zip_contains_repo(p)]

if zip_candidates:
    code_zip = zip_candidates[0]
    print("CODE_ZIP:", code_zip)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(REPO_DIR)
elif legacy_zip_candidates:
    code_zip = legacy_zip_candidates[0]
    print("LEGACY_CODE_ZIP:", code_zip)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(REPO_DIR)
elif scanned_repo_zips:
    code_zip = scanned_repo_zips[0]
    print("SCANNED_REPO_ZIP:", code_zip)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(REPO_DIR)
elif repo_input_candidates:
    src = repo_input_candidates[0]
    print("CODE_SRC:", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        "No repo code input found. Add a Kaggle dataset containing iu_xray_kaggle_code_controlled.zip "
        "or any zip whose root contains train_ablation.py. See printed /kaggle/input list above."
    )

nested_train = list(REPO_DIR.glob("**/train_ablation.py"))
if nested_train and not (REPO_DIR / "train_ablation.py").exists():
    nested_root = nested_train[0].parent
    print("Detected nested repo root:", nested_root)
    for item in nested_root.iterdir():
        target = REPO_DIR / item.name
        if target.exists():
            continue
        shutil.move(str(item), str(target))

os.chdir(REPO_DIR)
for rel in ["train_proposed.py", "train_ablation.py", "train_single_gpu.py", "requirements.txt"]:
    if not os.path.exists(rel):
        raise FileNotFoundError(f"Missing {rel} after code setup in {REPO_DIR}")
print("REPO_DIR:", REPO_DIR)
print("repo files:", os.listdir(".")[:40])

print("\n=== Find IU-Xray data ===")
print("requested DATA_ROOT:", DATA_ROOT)

csv_candidates = []
if os.path.exists(DATA_ROOT + "/data/v8_clean.csv"):
    csv_candidates.append(DATA_ROOT + "/data/v8_clean.csv")
if os.path.exists(DATA_ROOT + "/v8_clean.csv"):
    csv_candidates.append(DATA_ROOT + "/v8_clean.csv")

if not csv_candidates:
    csv_candidates = sorted(glob.glob("/kaggle/input/**/v8_clean.csv", recursive=True))

print("v8_clean.csv candidates:", csv_candidates[:20])
if not csv_candidates:
    raise FileNotFoundError("Cannot find v8_clean.csv under /kaggle/input. Check the IU-Xray dataset input path printed above.")

CSV_PATH = csv_candidates[0]
csv_parent = os.path.dirname(CSV_PATH)
img_candidates = []
for candidate in [
    os.path.join(csv_parent, "images_384"),
    os.path.join(os.path.dirname(csv_parent), "images_384"),
    os.path.join(os.path.dirname(csv_parent), "data", "images_384"),
]:
    if os.path.isdir(candidate):
        img_candidates.append(candidate)
if not img_candidates:
    img_candidates = sorted(glob.glob("/kaggle/input/**/images_384", recursive=True))

print("images_384 candidates:", img_candidates[:20])
if not img_candidates:
    raise FileNotFoundError("Cannot find images_384 under /kaggle/input. Check the IU-Xray dataset input path printed above.")

IMG_DIR = img_candidates[0]
DATA_ROOT = os.path.commonpath([CSV_PATH, IMG_DIR])
print("DATA_ROOT_DETECTED:", DATA_ROOT)
print("CSV_PATH:", CSV_PATH, os.path.exists(CSV_PATH))
print("IMG_DIR:", IMG_DIR, os.path.exists(IMG_DIR))
print("num images:", len(glob.glob(IMG_DIR + "/*")))

print("\n=== Install requirements and check GPUs ===")
subprocess.check_call(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.call(["nvidia-smi"])

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Kaggle accelerator must be GPU T4 x2 for this dual-run launcher.")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print("gpu", idx, props.name, "vram_gb", round(props.total_memory / 1024**3, 2))

RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

run_manifest = {
    "run_id": RUN_ID,
    "mode": MODE,
    "embed_config": EMBED_CONFIG,
    "config": CONFIGS[EMBED_CONFIG],
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS,
    "eval_every": EVAL_EVERY,
    "freeze_ep": FREEZE_EP,
    "cluster_start": CLUSTER_START,
    "clinical_start": CLINICAL_START,
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "csv_path": CSV_PATH,
    "img_dir": IMG_DIR,
    "resume_mode": RESUME_MODE,
}
(RUN_ROOT / "run_config.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

def find_resume_checkpoint(method: str) -> str | None:
    if RESUME_MODE.lower() == "off":
        return None
    if RESUME_MODE.lower() != "auto":
        return RESUME_MODE

    candidates = []
    for root in sorted(glob.glob("/kaggle/input/*")):
        patterns = [
            f"{root}/**/{RUN_ID}/{method}/last.pt",
            f"{root}/**/{RUN_ID}/{method}/best.pt",
            f"{root}/**/{EMBED_CONFIG}_{MODE}/{method}/last.pt",
            f"{root}/**/{method}/last.pt",
        ]
        for pattern in patterns:
            candidates.extend(glob.glob(pattern, recursive=True))
    candidates = sorted(set(candidates), key=lambda x: ("last.pt" not in x, x))
    return candidates[0] if candidates else None

common_args = [
    "--csv_path", CSV_PATH,
    "--img_dir", IMG_DIR,
    "--epochs", EPOCHS,
    "--batch_size", BATCH_SIZE,
    "--grad_accum", GRAD_ACCUM,
    "--eval_every", EVAL_EVERY,
    "--freeze_ep", FREEZE_EP,
    "--num_workers", NUM_WORKERS,
    "--seed", SEED,
    "--split_seed", SPLIT_SEED,
    "--image_model", CONFIGS[EMBED_CONFIG]["image_model"],
    "--text_model", CONFIGS[EMBED_CONFIG]["text_model"],
]

strict_cmd = [
    "python", "-u", "train_ablation.py", "study_multiview_strict",
    "--out_dir", str(RUN_ROOT / "strict"),
    *common_args,
]
proposed_cmd = [
    "python", "-u", "train_ablation.py", "proposed",
    "--out_dir", str(RUN_ROOT / "proposed"),
    *common_args,
    "--cluster_start", CLUSTER_START,
    "--clinical_start", CLINICAL_START,
]

commands = [
    ("strict", "0", strict_cmd),
    ("proposed", "1", proposed_cmd),
]

processes = []
start_time = time.time()
for name, gpu, cmd in commands:
    resume = find_resume_checkpoint(name)
    if resume:
        cmd = [*cmd, "--resume", resume]
    out_dir = RUN_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu
    log_path = LOG_ROOT / f"{RUN_ID}_{name}.log"
    launcher_info = {
        "name": name,
        "gpu": gpu,
        "cmd": cmd,
        "log_path": str(log_path),
        "out_dir": str(out_dir),
        "resume": resume or "",
    }
    (out_dir / "launcher_config.json").write_text(json.dumps(launcher_info, indent=2), encoding="utf-8")
    print("\n=== Launch", name, "on GPU", gpu, "===")
    print("OUT_DIR:", out_dir)
    print("LOG:", log_path)
    print("RESUME:", resume or "none")
    print("CMD:", " ".join(cmd))
    log_f = open(log_path, "w", encoding="utf-8")
    log_f.write(json.dumps(launcher_info, indent=2) + "\n\n")
    log_f.flush()
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env, text=True)
    processes.append((name, proc, log_f, log_path))

failed = []
log_positions = {name: 0 for name, _, _, _ in processes}
print("\n=== Streaming training logs every 10 seconds ===", flush=True)
while True:
    all_done = True
    for name, proc, log_f, log_path in processes:
        if proc.poll() is None:
            all_done = False
        log_f.flush()
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as reader:
                reader.seek(log_positions[name])
                new_text = reader.read()
                log_positions[name] = reader.tell()
        except FileNotFoundError:
            new_text = ""
        if new_text:
            for line in new_text.rstrip().splitlines():
                print(f"[{name}] {line}", flush=True)
    if all_done:
        break
    time.sleep(10)

for name, proc, log_f, log_path in processes:
    ret = proc.wait()
    log_f.close()
    print("FINISHED:", name, "returncode", ret, "log", log_path)
    if ret != 0:
        failed.append((name, ret, str(log_path)))

elapsed = round(time.time() - start_time, 1)
print("elapsed_seconds:", elapsed)

print("\n=== Output check ===")
for method in ["strict", "proposed"]:
    out_dir = RUN_ROOT / method
    print("\n", method, out_dir)
    if out_dir.exists():
        print(os.listdir(out_dir))
    for filename in ["train.log", "progress.log", "history.csv", "latest_metrics.json", "test_results.json", "last.pt", "best.pt", "launcher_config.json"]:
        path = out_dir / filename
        print(filename, path.exists(), round(path.stat().st_size / 1024**2, 3) if path.exists() else None)
    log_path = LOG_ROOT / f"{RUN_ID}_{method}.log"
    print("launcher_log", log_path.exists(), round(log_path.stat().st_size / 1024**2, 3) if log_path.exists() else None)

print("\n=== Zip outputs ===")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
zip_base = str(OUTPUT_ZIP).replace(".zip", "")
shutil.make_archive(zip_base, "zip", "/kaggle/working", "runs")
with zipfile.ZipFile(OUTPUT_ZIP, "a", compression=zipfile.ZIP_DEFLATED) as zf:
    for log_file in LOG_ROOT.glob(f"{RUN_ID}_*.log"):
        zf.write(log_file, arcname=f"logs/{log_file.name}")
print("Created:", OUTPUT_ZIP)
print("size_mb:", round(OUTPUT_ZIP.stat().st_size / 1024**2, 2))

if failed:
    raise RuntimeError(f"Some runs failed: {failed}")
print("DONE")

=== Run config ===
MODE: probe
EMBED_CONFIG: vit_bioclinicalbert
BATCH_SIZE: 6
GRAD_ACCUM: 5
EPOCHS: 3
EVAL_EVERY: 1
FREEZE_EP: 1
CLUSTER_START: 2
CLINICAL_START: 2
SEED: 42
SPLIT_SEED: 42
RESUME_MODE: off
RUN_ID: vit_bioclinicalbert_probe_seed42_split42
CONFIG: {
  "image_model": "google/vit-base-patch16-384",
  "text_model": "emilyalsentzer/Bio_ClinicalBERT"
}

=== Kaggle inputs ===

 /kaggle/input/datasets
['diplailai']

=== Setup code ===
expected code zip name: iu_xray_kaggle_code_controlled.zip
matching code zips: []
legacy code zips: []
all input zips: []
repo directory candidates: ['/kaggle/input/datasets/diplailai/iu-xray-kaggle-code-controlled']
CODE_SRC: /kaggle/input/datasets/diplailai/iu-xray-kaggle-code-controlled
REPO_DIR: /kaggle/working/finetune_IU_Xray
repo files: ['train.py', 'train_single_gpu.py', 'benchmark_suite', 'README.md', 'requirements.txt', 'train_proposed.py', 'train_ablation.py', 'src', 'data_processing']

=== Find IU-Xray data ===
requested DATA_ROOT: /ka

In [4]:
!tail -n 80 /kaggle/working/logs/swin_bioclinicalbert_probe_seed42_split42_strict.log
!tail -n 80 /kaggle/working/logs/swin_bioclinicalbert_probe_seed42_split42_proposed.log

tail: cannot open '/kaggle/working/logs/swin_bioclinicalbert_probe_seed42_split42_strict.log' for reading: No such file or directory
tail: cannot open '/kaggle/working/logs/swin_bioclinicalbert_probe_seed42_split42_proposed.log' for reading: No such file or directory
